# 06 UPDATE and DELETE

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Change and remove rows in Delta tables with <code>UPDATE</code> and <code>DELETE</code> (SQL and Python), read their metrics in the history, and understand how Delta implements them on immutable Parquet files: copy-on-write, and <b>deletion vectors</b>.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Correcting bad data, applying GDPR "right to be forgotten" requests, and soft-deleting records are everyday tasks. On plain Parquet each one means rewriting folders by hand. On Delta it's one statement. Knowing how much data an <code>UPDATE</code> rewrites, and why deletion vectors make it cheaper, helps you design efficient tables.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 How row-level changes work on immutable files</b>
<ul><li><b>Copy-on-write</b> (classic): find the files containing matching rows, write <b>new</b> files with the changes applied, and commit <code>remove</code> (old files) + <code>add</code> (new files). A one-row update can rewrite a whole 1 GB file</li><li><b>Deletion vectors</b> (merge-on-read): instead of rewriting a file, write a small <b>deletion vector</b> marking which rows of the file are deleted (or replaced). Updated rows are written to a new small file. Readers skip the marked rows. Rewrites happen later, during <code>OPTIMIZE</code> or <code>REORG</code></li><li>Both are atomic: the change is one commit</li><li>Old files stay on storage until <code>VACUUM</code>, so time travel and <code>RESTORE</code> work</li></ul>
</div>

| | Copy-on-write | Deletion vectors |
|---|---|---|
| Write cost | High (rewrite whole files) | Low (small DV file + changed rows) |
| Read cost | None extra | Small (apply the DVs while reading) |
| Enabled by | Default in open-source Delta | Default for new tables on recent Databricks runtimes (`delta.enableDeletionVectors`) |

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A telecom company receives 200 GDPR deletion requests a day for a 5 TB call-records table. With copy-on-write, each daily <code>DELETE</code> rewrote about 600 GB of files because the customers were spread across many files, and it took 2 hours. With deletion vectors enabled, the same delete wrote a few MB of deletion vectors and finished in 4 minutes. A weekly <code>OPTIMIZE</code> then compacts the files and physically removes the deleted rows.
</div>

## Syntax

```sql
UPDATE t SET amount = amount * 1.1, updated_at = current_timestamp() WHERE country = 'IN';
DELETE FROM t WHERE created < '2022-01-01';
```

```python
from delta.tables import DeltaTable
dt = DeltaTable.forName(spark, "cat.sch.t")
dt.update(condition="country = 'IN'", set={"amount": "amount * 1.1"})
dt.delete("created < '2022-01-01'")
```

## Setup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates an <code>accounts</code> table with 10,000 rows spread over 4 files.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
4 files and 10,000 rows.
</div>

In [0]:
DB = "workspace.delta_lake"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}")

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

table = f"{DB}.accounts_dml"
(
    spark.range(10_000).repartition(4)
    .select(
        F.col("id").alias("account_id"),
        F.element_at(F.array(*[F.lit(c) for c in ["IN", "UK", "AE", "US"]]), (F.col("id") % 4 + 1).cast("int")).alias("country"),
        (F.col("id") % 5000 / 10).alias("balance"),
        F.when(F.col("id") % 50 == 0, "closed").otherwise("active").alias("status"),
    )
    .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table)
)
spark.sql(f"ALTER TABLE {table} SET TBLPROPERTIES ('delta.enableDeletionVectors' = 'false')")   # start with copy-on-write

def metrics(n=1):
    return spark.sql(f"DESCRIBE HISTORY {table} LIMIT {n}").select("version", "operation", "operationMetrics")

print("files:", spark.sql(f"DESCRIBE DETAIL {table}").first()["numFiles"], "| rows:", spark.table(table).count())

## 1. UPDATE in SQL

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Adds 5% interest to active Indian accounts and stamps them, then reads the operation metrics.<br><br>
<b>Why it matters</b><br>The metrics tell you how much work an update really did: <code>numUpdatedRows</code> (rows changed), <code>numCopiedRows</code> (unchanged rows that had to be rewritten), and the files removed and added. A high <code>numCopiedRows</code> is the cost of copy-on-write.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
About 2,400 updated rows. Because the Indian accounts are spread over all 4 files, all 4 files are rewritten and <code>numCopiedRows</code> is about 7,600.
</div>

In [0]:
spark.sql(f"""
    UPDATE {table}
    SET balance = round(balance * 1.05, 2)
    WHERE country = 'IN' AND status = 'active'
""")
display(metrics())

## 2. DELETE in SQL

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Deletes closed accounts and checks the counts.<br><br>
<b>Why it matters</b><br>Same mechanism: files with matching rows are rewritten without them. A <code>DELETE</code> with no <code>WHERE</code> clause removes all rows, but it's a metadata-only operation (the files are just marked as removed).
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
200 closed accounts deleted, 9,800 rows remaining, with metrics showing <code>numDeletedRows = 200</code>.
</div>

In [0]:
spark.sql(f"DELETE FROM {table} WHERE status = 'closed'")
display(metrics())
print("rows now:", spark.table(table).count())

## 3. The Python DeltaTable API

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Runs an update and a delete with <code>DeltaTable.forName(...)</code>.<br><br>
<b>Why it matters</b><br>In Python pipelines, the <code>DeltaTable</code> API avoids building SQL strings. Conditions and values are SQL expressions as strings, or Column objects.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
UK balances under 10 are set to 0, and accounts with a zero balance are deleted. The history shows an <code>UPDATE</code> then a <code>DELETE</code>.
</div>

In [0]:
dt = DeltaTable.forName(spark, table)
dt.update(condition=(F.col("country") == "UK") & (F.col("balance") < 10), set={"balance": F.lit(0.0)})
dt.delete(F.col("balance") == 0)
display(metrics(2))

## 4. Deletion vectors: much less to write

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Enables deletion vectors, runs a small delete, and compares the metrics with the copy-on-write delete.<br><br>
<b>Why it matters</b><br>With deletion vectors, the delete adds a small DV file per affected data file instead of rewriting the files. <code>numDeletionVectorsAdded &gt; 0</code> and <code>numCopiedRows = 0</code> are the signs. Writes get much faster, at a small cost to reads until the next <code>OPTIMIZE</code>.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
New tables on recent Databricks runtimes have deletion vectors on by default. Enabling them upgrades the table protocol (reader version 3, writer version 7).
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The delete removes about 100 rows with <code>numDeletionVectorsAdded</code> greater than 0, <code>numCopiedRows</code> equal to 0, and no files removed.
</div>

In [0]:
spark.sql(f"ALTER TABLE {table} SET TBLPROPERTIES ('delta.enableDeletionVectors' = 'true')")
spark.sql(f"DELETE FROM {table} WHERE account_id % 97 = 0")
display(metrics())

## 5. Conditional updates and multiple columns

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Uses <code>CASE WHEN</code> inside <code>UPDATE</code> to apply different changes in one pass.<br><br>
<b>Why it matters</b><br>One <code>UPDATE</code> with <code>CASE</code> rewrites the files once. Three separate updates would rewrite them three times.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Balances changed differently per country in one <code>UPDATE</code> commit. A sample per country is shown.
</div>

In [0]:
spark.sql(f"""
    UPDATE {table}
    SET balance = CASE country
                    WHEN 'AE' THEN balance + 10
                    WHEN 'US' THEN balance - 5
                    ELSE balance
                  END,
        status = CASE WHEN balance > 450 THEN 'premium' ELSE status END
    WHERE country IN ('AE', 'US')
""")
spark.table(table).groupBy("country", "status").count().orderBy("country", "status").show()

## 6. Updating from another table

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Explains why <code>UPDATE</code> can't join to another table, and shows the <code>MERGE</code> form.<br><br>
<b>Why it matters</b><br>Delta's <code>UPDATE</code> only uses a condition on the target table. To update rows <b>using values from another table</b> (for example, a corrections file), use <code>MERGE ... WHEN MATCHED THEN UPDATE</code> (lesson 07).
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Accounts 2 and 3 show their corrected balances, 999.0 and 888.0.
</div>

In [0]:
spark.createDataFrame([(2, 999.0), (3, 888.0)], "account_id BIGINT, new_balance DOUBLE").createOrReplaceTempView("corrections")
spark.sql(f"""
    MERGE INTO {table} t
    USING corrections c ON t.account_id = c.account_id
    WHEN MATCHED THEN UPDATE SET t.balance = c.new_balance
""")
spark.sql(f"SELECT account_id, balance FROM {table} WHERE account_id IN (2, 3) ORDER BY account_id").show()

## Under the hood

```
UPDATE t SET x = ... WHERE cond   (copy-on-write)
  1. scan: find files containing rows that match cond (data skipping uses file stats)
  2. rewrite each such file: changed rows + unchanged rows ("copied rows")
  3. commit: remove old files, add new files

DELETE ... (deletion vectors)
  1. scan: find matching rows and their positions
  2. write a deletion vector (bitmap of row positions) per affected file
  3. commit: add the same files again, now with a deletionVector reference
  readers: read the file, skip rows marked in its DV
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Action</span> `UPDATE` and `DELETE` run immediately and create one version each.

The key to cheap DML is **touching few files**: if the rows you update are clustered together (by partition, Z-order or liquid clustering), fewer files match and less is rewritten. Lesson 10 covers layout.

In [0]:
spark.sql(f"DESCRIBE HISTORY {table}").select("version", "operation", "operationMetrics.numCopiedRows", "operationMetrics.numDeletionVectorsAdded").show()

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {table}")

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: a tiny update takes a long time</b><br>
Copy-on-write rewrote large files because matching rows were spread everywhere. Enable deletion vectors, cluster the data by the filter column, or batch changes into one statement.<br><br>
**⛔ Problem: <code>UPDATE ... FROM other_table</code> is not supported**<br>Use <code>MERGE INTO ... WHEN MATCHED THEN UPDATE</code>.<br><br>
<b>⛔ Problem: deleted data still on storage</b><br><code>DELETE</code> only removes rows from the current version. For compliance, follow with <code>VACUUM</code> after the retention period (and <code>REORG TABLE ... APPLY (PURGE)</code> when using deletion vectors).<br><br>
**⛔ Problem: <code>ConcurrentAppendException</code> / conflicts during updates**<br>Another job modified the same files. Run the jobs in sequence or narrow the conditions.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. How does Delta perform an <code>UPDATE</code> when Parquet files are immutable?</b><br>
By default with copy-on-write: it finds the files containing matching rows, writes new versions of those files with the changes, and commits removing the old files and adding the new ones atomically. With deletion vectors, it marks the old rows as deleted in a small side file and writes only the changed rows, which is much cheaper.<br><br>

<b>🎤 2. What are deletion vectors?</b><br>
A Delta feature that records deleted or updated row positions per data file in a compact bitmap, instead of rewriting the file. Readers filter those rows out. The files are physically rewritten later, by <code>OPTIMIZE</code> or <code>REORG</code>.<br><br>

<b>🎤 3. Does <code>DELETE</code> physically remove the data?</b><br>
Not immediately. The rows disappear from the current version, but older files stay for time travel until <code>VACUUM</code> removes files older than the retention period. With deletion vectors, <code>REORG TABLE ... APPLY (PURGE)</code> rewrites files to physically drop the rows.<br><br>

<b>🎤 4. How do you update a table using values from another table?</b><br>
With <code>MERGE INTO target USING source ON key WHEN MATCHED THEN UPDATE SET ...</code>, because Delta's <code>UPDATE</code> can't join to another table.<br><br>

<b>🎤 5. How can you see how expensive a DML operation was?</b><br>
In <code>DESCRIBE HISTORY</code>, the <code>operationMetrics</code> show updated, deleted and copied rows, files added and removed, and deletion vectors added.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A user must be removed from a Delta table for a GDPR request, and the data must no longer be recoverable after the standard retention period. Which combination is correct?</b><br>
A. <code>DELETE</code> only<br>
B. <code>DELETE</code>, then <code>VACUUM</code> once the retention period has passed<br>
C. <code>TRUNCATE TABLE</code><br>
D. <code>DROP TABLE</code> and recreate it<br>
<details><summary><b>Show answer</b></summary><b>B.</b> <code>DELETE</code> alone keeps old files for time travel. <code>VACUUM</code> removes them after the retention period.</details><br><br>

<b>Q2. Which Delta feature lets a <code>DELETE</code> avoid rewriting whole data files?</b><br>
A. Change data feed<br>
B. Deletion vectors<br>
C. Column mapping<br>
D. Liquid clustering<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Create a table <code>subscriptions</code> with 5,000 rows: <code>sub_id</code>, <code>plan</code> (basic/pro), <code>price</code>, <code>active</code> (boolean)</li><li>In one <code>UPDATE</code>, raise basic prices by 10% and pro prices by 5%</li><li>Delete inactive subscriptions with copy-on-write, and record <code>numCopiedRows</code></li><li>Enable deletion vectors, delete the subscriptions whose <code>sub_id</code> ends in 7, and compare the metrics</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
subs = f"{DB}.subscriptions"

# 1. Table without deletion vectors
(
    spark.range(5000).repartition(4)
    .select(F.col("id").alias("sub_id"),
            F.when(F.col("id") % 3 == 0, "pro").otherwise("basic").alias("plan"),
            F.when(F.col("id") % 3 == 0, 20.0).otherwise(10.0).alias("price"),
            (F.col("id") % 10 != 0).alias("active"))
    .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(subs)
)
spark.sql(f"ALTER TABLE {subs} SET TBLPROPERTIES ('delta.enableDeletionVectors' = 'false')")

# 2. One UPDATE with CASE
spark.sql(f"UPDATE {subs} SET price = CASE plan WHEN 'basic' THEN price * 1.10 ELSE price * 1.05 END")

# 3. Copy-on-write delete
spark.sql(f"DELETE FROM {subs} WHERE NOT active")
print("copy-on-write:", spark.sql(f"DESCRIBE HISTORY {subs} LIMIT 1").first()["operationMetrics"])

# 4. Deletion-vector delete
spark.sql(f"ALTER TABLE {subs} SET TBLPROPERTIES ('delta.enableDeletionVectors' = 'true')")
spark.sql(f"DELETE FROM {subs} WHERE sub_id % 10 = 7")
print("deletion vectors:", spark.sql(f"DESCRIBE HISTORY {subs} LIMIT 1").first()["operationMetrics"])
spark.sql(f"DROP TABLE {subs}")

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li><code>UPDATE</code> and <code>DELETE</code> are single atomic commits on Delta tables</li><li>Copy-on-write rewrites every file that contains a matching row (<code>numCopiedRows</code> shows the overhead)</li><li>Deletion vectors mark rows instead of rewriting files. Much faster writes, compacted later</li><li>Combine changes with <code>CASE</code> to rewrite files once</li><li>To update from another table, use <code>MERGE</code></li><li>Deleted rows remain in old files until <code>VACUUM</code></li></ul>
</div>

| Task | Code |
|---|---|
| Update | `UPDATE t SET c = expr WHERE cond` |
| Delete | `DELETE FROM t WHERE cond` |
| Python | `DeltaTable.forName(spark, t).update(cond, {"c": expr})`, `.delete(cond)` |
| Deletion vectors | `ALTER TABLE t SET TBLPROPERTIES ('delta.enableDeletionVectors' = true)` |
| Cost | `DESCRIBE HISTORY t` → `operationMetrics` |

## Git commit

```
git add 03_delta_lake/06_update_delete.ipynb
git commit -m "add 03_06 update delete notebook"
```